# Audit the facility assessment evidence

Use the branch linkage and service verification checkpoints to prepare dated,
scope-aware evidence about external assessment and patient experience before lookup.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

# Fill this in if the notebook's working directory is outside the repository.
REPO_ROOT = os.environ.get("DATATHON_REPO_ROOT") or None
cwd = Path.cwd().resolve()
ROOT = Path(REPO_ROOT).expanduser().resolve() if REPO_ROOT else next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "non-ml" / "config" / "project.json").is_file()),
    None,
)
if ROOT is None or not (ROOT / "non-ml" / "config" / "project.json").is_file():
    raise RuntimeError("Set REPO_ROOT to the directory that contains non-ml/config/project.json.")
sys.path.insert(0, str(ROOT / "non-ml" / "src"))

from facility_tool.workspace import Store, read_config, show

config = read_config(ROOT)
MODE = os.environ.get("DATATHON_MODE", config["input_mode"])
STORE = Store(ROOT, MODE, config["as_of"])
print(f"Input mode: {MODE} | Evidence date: {config['as_of']}")

## What an assessment can establish

India's NQAS measures service provision, patient rights, inputs, support services,
clinical care, infection control, quality management and outcomes. The saved
directory gives certification status and dates. NABH has different accreditation
programs and supports the named branch and scope on a current certificate.
Record these signals separately from clinical outcomes and service readiness.
The WHO HHFA measures availability, readiness and quality independently.
[NQAS standards](https://qps.nhsrcindia.org/national-quality-assurance-standards) ·
[NABH hospital accreditation](https://nabh.co/programmes/hospitals-accreditation-programme-hco/) ·
[WHO facility assessment](https://www.who.int/data/data-collection-tools/harmonized-health-facility-assessment/introduction)

In [ ]:
from facility_tool.quality import build_quality_profiles

evidence, profiles = build_quality_profiles(
    STORE.read("records"), STORE.read("links"), STORE.read("verified_claims"),
    STORE.read("service_verification"), config["as_of"],
)
STORE.write("assessment_evidence", evidence)
STORE.write("quality_profiles", profiles)
show(evidence.groupby(["programme", "assessment_status", "link_status"]).size().rename("records").reset_index())
show(profiles.groupby(["service", "quality_assessment_status"]).size().rename("branches").reset_index(), limit=35)
show(profiles.loc[profiles["quality_assessed"].eq(True)], limit=10)

## Attach patient feedback to the exact branch

Optionally provide a permitted source export of linked aggregates. A rated branch
keeps its original metric, reporting window, sample size and source. Missing ratings
stay unknown. Read the source and scope before comparing them. The branch review
search link works independently of the ratings import.
[How to find and use patient feedback](../QUALITY_METHOD.md)

In [ ]:
from facility_tool.patient_experience import prepare_patient_experience

rating_path = os.environ.get("DATATHON_PATIENT_RATINGS")
ratings = pd.read_csv(rating_path, dtype={"facility_id": str}) if rating_path and MODE != "demo" else pd.DataFrame()
patient_experience = prepare_patient_experience(STORE.read("entities"), ratings, config["as_of"])
STORE.write("patient_experience", patient_experience)
show(patient_experience["patient_experience_status"].value_counts().rename("branches").reset_index())